# 01 — Dataset Inspection and Whisper Test

**Purpose:** inspect Nepali–English code-switched audio and test Whisper Large V3 Turbo with automatic language detection versus forced Nepali. This continues the earlier exploratory work in a notebook that can run independently from a fresh Colab session.

**Scope:** the same five code-switched validation clips in both decoding modes. The original exploratory test clips remain previously exposed. These pilot results do not establish a CMI trend or constitute the final three-model benchmark.

The notebook follows four topics:

1. **Environment Setup** — dependencies, imports, experiment settings, and GPU checks.
2. **Dataset Inspection** — reference preparation, loading, CS sample selection, and audio validation/playback.
3. **Whisper Test** — model loading and automatic versus forced-Nepali transcription.
4. **Pilot Results** — aggregate WER/CER and a downloadable results/environment archive.

**In Google Colab:** select **Runtime → Change runtime type → GPU** (T4 is sufficient for this pilot), then run cells from top to bottom. If installation requests a restart, restart and resume at **1.2 Import Libraries**. Use Colab's table of contents to navigate the headings.

Downloads and inference happen on the Colab runtime. No Google Drive mount or Hugging Face token is required for these public resources. Download the final ZIP before the runtime ends; see the [Colab FAQ](https://research.google.com/colaboratory/faq.html).


**Beginning work retained:** first-record keys/metadata; all three transcript formats; an unfiltered five-record preview; audio shape, dtype, sample rate, duration, and amplitude checks; first-record playback; seeded CS-only sampling; choosing sample 3 to replay; preparing Whisper inputs; and forced-Nepali transcription. Repeated extraction/import cells are consolidated. The original notebook remains unchanged, and its saved outputs are not presented as new results.


## 1. Environment Setup

Prepare the environment and record the settings needed to repeat this pilot.


### 1.1 Install Dependencies

Keep Colab's GPU-enabled PyTorch installation. Direct dependencies below are pinned; the final export captures all installed package versions. WAV audio is decoded with SoundFile, so TorchCodec/FFmpeg are not needed for this pilot.


Unused pandas, matplotlib, and Accelerate imports from the original exploration are omitted. No plot or training step was present in that notebook.

**Colab dependency notices:** the pinned Transformers/Datasets releases can downgrade huggingface-hub and fsspec, causing pip to report conflicts with preinstalled diffusers, gradio, gradio-client, and gcsfs. This notebook does not use those four packages or mount Google Drive. These notices alone do not demonstrate that the ASR imports work or fail. After the first installation, restart the Colab session and resume at **1.2 Import Libraries**. If imports fail, stop and retain the full traceback. Do not upgrade huggingface-hub/fsspec independently: the pinned ASR packages constrain their versions. A compatible import and actual pilot run are still required; the broader Colab environment remains inconsistent for the listed packages.

**Restart is required after installing or changing packages**, even if pip does not request it. Save your notebook first, including any completed `reviews` list. Choose **Runtime → Restart session**, then rerun from section 1.2 onward without rerunning installation. Restarting clears variables; saved cell edits remain. Mixing old imported modules with newly installed files can cause missing Hugging Face attributes.


In [ ]:
%pip install -q "transformers==4.57.1" "huggingface-hub==0.36.2" "datasets==4.4.1" "librosa==0.11.0" "soundfile==0.13.1" "jiwer==4.0.0"


### 1.2 Import Libraries

Import the tools for dataset streaming, audio playback, inference, scoring, and export.


In [ ]:
import copy
import hashlib
import io
import json
import math
import platform
import subprocess
import sys
import unicodedata
import zipfile
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import huggingface_hub
from huggingface_hub import constants as hub_constants

# Package changes need a session restart to clear already imported modules.
assert version('huggingface-hub') == huggingface_hub.__version__ == '0.36.2', (
    'Run section 1.1, restart the Colab session, then resume at section 1.2.'
)
assert hasattr(hub_constants, 'HF_HUB_ENABLE_HF_TRANSFER'), (
    'Mixed Hugging Face modules are still loaded. Restart the Colab session before continuing.'
)
assert version('transformers') == '4.57.1', 'Run section 1.1, restart, then resume at 1.2.'

import numpy as np
import soundfile as sf
import torch
from huggingface_hub import snapshot_download
from datasets import load_dataset
from IPython.display import Audio, display
from jiwer import cer, wer
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor



### 1.3 Set Experiment Parameters

Dataset and model revisions were resolved on October 16, 2026. Keep them unchanged to reuse the same files. Use validation data for development; five buffered-shuffle samples are not a representative CMI sample.


In [ ]:
DATASET_ID = 'saileshbro/nepali-cs-asr'
DATASET_REVISION = 'dd00ca41dd38774cd6097c372c16939d7f3746df'
MODEL_ID = 'openai/whisper-large-v3-turbo'
MODEL_REVISION = '41f01f3fe87f28c78e2fbf8b568835947dd65ed9'
SPLIT = 'validation'
SEED = 42
SAMPLE_COUNT = 5
SHUFFLE_BUFFER = 1000
MAX_NEW_TOKENS = 440


SAMPLE_NUMBER = 3
if not isinstance(SAMPLE_COUNT, int) or SAMPLE_COUNT < 1:
    raise ValueError("SAMPLE_COUNT must be a positive integer.")
if not isinstance(SAMPLE_NUMBER, int) or not 1 <= SAMPLE_NUMBER <= SAMPLE_COUNT:
    raise ValueError("SAMPLE_NUMBER must be between 1 and SAMPLE_COUNT.")


### 1.4 Check GPU and Package Versions

Confirm that the notebook has GPU access and print its runtime details. Stop here if a GPU is unavailable.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Runtime → Change runtime type, reconnect, and rerun.')
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
DEVICE = 'cuda:0'
DTYPE = torch.float16
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print({name: version(name) for name in ['transformers', 'datasets', 'huggingface-hub', 'fsspec', 'librosa', 'soundfile', 'jiwer']})


## 2. Dataset Inspection

Prepare the reference text, select code-switched validation clips, and inspect their audio and metadata before running Whisper.


### 2.1 Prepare Reference Text and Check Scoring

Apply NFC, lowercase, Devanagari-digit conversion, punctuation-to-space, and whitespace collapse to both reference and hypothesis. Preserve raw strings in the export. WER uses whitespace tokens; CER uses Unicode code points after removing spaces. These checks ensure that empty hypotheses remain errors and that Latin words are not silently equated with Devanagari transliterations.


In [ ]:
def normalize_text(text):
    text = unicodedata.normalize('NFC', text).lower()
    text = text.translate(str.maketrans('०१२३४५६७८९', '0123456789'))
    text = ''.join(' ' if unicodedata.category(char).startswith('P') else char for char in text)
    return ' '.join(text.split())

assert normalize_text('Hello,  SCHOOL!') == 'hello school'
assert normalize_text('नेपालमा १२ school छन्।') == 'नेपालमा 12 school छन्'
assert normalize_text('cafe\u0301') == normalize_text('café')
assert normalize_text('school') != normalize_text('स्कुल')
assert normalize_text(' ' + chr(9) + chr(10)) == ''
assert wer(['a b', 'c d'], ['a', 'c d e']) == 0.5
assert wer('a b', '') == 1.0
assert wer('a', 'a b c') == 2.0
assert cer('ab', 'a') == 0.5
print('Normalization and scoring self-checks passed.')


### 2.2 Load the Dataset and Inspect Its Structure

Print the streamed dataset, retained columns, and schema. Disable all media decoding before selecting columns so unused video does not require a decoder. Include all three transcript formats for inspection. Derived renderings may be stale according to the dataset card; only canonical transcription is used for scoring.


In [ ]:
columns = [
    'audio', 'transcription', 'transcription_devanagari', 'transcription_romanized',
    'cmi', 'speech_type', 'speaker_id', 'video_id', 'start_ms', 'end_ms', 'duration_s',
]
base_stream = load_dataset(
    DATASET_ID, name='default', split=SPLIT, revision=DATASET_REVISION, streaming=True,
)
base_stream = base_stream.decode(False).select_columns(columns)
print(base_stream)
print('\nCOLUMNS:', base_stream.column_names)
print('\nFEATURES:', base_stream.features)


### 2.3 Preview Five Records Before Filtering

Retain your initial unfiltered inspection. Records may be ne, en, or cs; do not assume they are all code-switched. This preview does not consume the separate CS iterator used for inference.


In [ ]:
unfiltered_preview = list(base_stream.take(5))
if not unfiltered_preview:
    raise ValueError('The selected split contains no records.')
for number, row in enumerate(unfiltered_preview, start=1):
    print(f'\nPREVIEW {number} | speech_type={row["speech_type"]} | CMI={row["cmi"]} | {row["duration_s"]}s')
    print(row['transcription'])
example = unfiltered_preview[0]


### 2.4 Inspect the First Record's Keys and Metadata

Keep the first unfiltered example distinct from the later filtered CS selection.


In [ ]:
print('KEYS:', list(example.keys()))
print('\nREFERENCE TRANSCRIPT:', example['transcription'])
print('\nCMI:', example['cmi'])
print('SPEECH TYPE:', example['speech_type'])
print('DURATION:', example['duration_s'], 'seconds')
print('CHANNEL:', example['speaker_id'])
print('PROVENANCE:', example['video_id'], example['start_ms'], example['end_ms'])


### 2.5 Compare All Three Transcript Formats

Retain the mixed-script, Devanagari, and romanized views. The latter two are display-only: they are derived renderings and are not independent ground truth.


In [ ]:
print('ORIGINAL / CANONICAL:')
print(example['transcription'])
print('\nDEVANAGARI — display only:')
print(example['transcription_devanagari'])
print('\nROMANIZED — display only:')
print(example['transcription_romanized'])


### 2.6 Decode Audio and Keep Its Reference Attached

One shared decoder replaces repeated extraction and global example/ex assignments. It checks provenance, audio, duration, reference, and CMI, then returns the mono float32 waveform together with its own metadata. Channel IDs do not establish individual speaker identities.


In [ ]:
def prepare_clip(row):
    video_id = row.get('video_id')
    start, end = row.get('start_ms'), row.get('end_ms')
    if (
        not isinstance(video_id, str) or not video_id.strip()
        or any(not isinstance(value, int) or isinstance(value, bool) for value in (start, end))
        or not 0 <= start < end
    ):
        raise ValueError('Missing or invalid video_id/start_ms/end_ms; inspect clip provenance.')
    key = f'{SPLIT}/{video_id}/{start}-{end}'
    reference, cmi = row.get('transcription'), row.get('cmi')
    if not isinstance(reference, str) or not normalize_text(reference):
        raise ValueError(f'Empty or invalid reference for {key}.')
    if isinstance(cmi, bool) or not isinstance(cmi, (int, float)) or not math.isfinite(cmi) or not 0 <= cmi <= 50:
        raise ValueError(f'Invalid bilingual CMI for {key}: {cmi}.')
    encoded = row.get('audio')
    if not isinstance(encoded, dict) or not isinstance(encoded.get('bytes'), bytes) or not encoded['bytes']:
        raise ValueError(f'Expected embedded WAV bytes for {key}; inspect the dataset format.')
    waveform, sample_rate = sf.read(io.BytesIO(encoded['bytes']), dtype='float32', always_2d=True)
    if sample_rate != 16000 or waveform.shape[0] == 0 or waveform.shape[1] == 0 or not np.isfinite(waveform).all():
        raise ValueError(f'Invalid or non-finite audio for {key}; expected nonempty 16 kHz audio.')
    channels = waveform.shape[1]
    audio = waveform.mean(axis=1)
    duration = len(audio) / sample_rate
    expected = row.get('duration_s')
    if (
        not 0 < duration <= 30
        or isinstance(expected, bool) or not isinstance(expected, (int, float)) or not math.isfinite(expected)
        or abs(duration - expected) > 0.02
        or abs(duration - (end - start) / 1000) > 0.02
    ):
        raise ValueError(f'Audio duration does not match metadata/provenance for {key}.')
    identity = {name: row[name] for name in columns if name != 'audio'}
    identity.update(
        clip_id=key, decoded_duration_s=duration, sample_rate=sample_rate, source_channels=channels,
        audio_sha256=hashlib.sha256(audio.tobytes()).hexdigest(),
    )
    return {'identity': identity, 'audio': audio}


def inspect_clip(clip):
    identity, audio = clip['identity'], clip['audio']
    print('CLIP:', identity['clip_id'])
    print('REFERENCE:', identity['transcription'])
    print('CMI:', identity['cmi'], '| SPEECH TYPE:', identity['speech_type'])
    print('Waveform shape:', audio.shape, '| dtype:', audio.dtype, '| type:', type(audio))
    print('Source channels:', identity['source_channels'], '| Sample rate:', identity['sample_rate'])
    print('Dataset duration:', identity['duration_s'], '| Decoded duration:', identity['decoded_duration_s'])
    print('Min:', float(audio.min()), '| Max:', float(audio.max()))
    print('Mean absolute amplitude:', float(np.abs(audio).mean()))
    display(Audio(audio, rate=identity['sample_rate']))


### 2.7 Inspect Waveform Diagnostics and Play the First Record

Retain the original amplitude/rate/shape checks and first-example playback. This exploratory preview is separate from the scored CS manifest.


In [ ]:
preview_clip = prepare_clip(example)
inspect_clip(preview_clip)


### 2.8 Filter to Code-Switched Speech and Shuffle Reproducibly

The original comment mentioned Nepali plus CS, while its filter actually retained only CS. The heading now matches the intended CS-only condition. Filter before sampling and leave the unfiltered stream intact.


In [ ]:
cs_stream = base_stream.filter(lambda row: row['speech_type'] == 'cs')
cs_stream = cs_stream.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)
selected_rows = list(cs_stream.take(SAMPLE_COUNT))
if len(selected_rows) != SAMPLE_COUNT:
    raise ValueError(f'Expected {SAMPLE_COUNT} CS clips, received {len(selected_rows)}.')
for number, row in enumerate(selected_rows, start=1):
    print(f'\nCS SAMPLE {number} | speech_type={row["speech_type"]} | CMI={row["cmi"]}')
    print(row['transcription'])


### 2.9 Prepare and Validate the Five CS Clips

Decode the selection using the same helper. Reject duplicate identities or decoded audio instead of replacing clips silently. Each clip keeps its own waveform and reference throughout inference. Keep zero-CMI CS labels visible for later annotation review.


In [ ]:
clips = []
seen_keys, seen_audio = set(), set()
for row in selected_rows:
    if row['speech_type'] != 'cs':
        raise ValueError('A non-CS record entered the pilot selection.')
    clip = prepare_clip(row)
    identity = clip['identity']
    if identity['clip_id'] in seen_keys or identity['audio_sha256'] in seen_audio:
        raise ValueError(f'Duplicate identity/audio: {identity["clip_id"]}. Inspect the selection.')
    seen_keys.add(identity['clip_id'])
    seen_audio.add(identity['audio_sha256'])
    clips.append(clip)
    print(f'{len(clips)} | {identity["clip_id"]} | CMI {identity["cmi"]} | {identity["decoded_duration_s"]:.3f}s')
    if identity['cmi'] == 0:
        print('CHECK: CS label with zero CMI; review token-language annotation before interpreting it.')


### 2.10 Choose a Sample and Replay Its Audio

Retain the sample_number=3 workflow. Change SAMPLE_NUMBER in section 1.3 and rerun this cell to inspect a different filtered clip. Its audio and reference come from the same object.


In [ ]:
if not isinstance(SAMPLE_NUMBER, int) or not 1 <= SAMPLE_NUMBER <= len(clips):
    raise ValueError('Choose SAMPLE_NUMBER between 1 and the number of selected CS clips.')
selected_clip = clips[SAMPLE_NUMBER - 1]
print('SELECTED CS SAMPLE:', SAMPLE_NUMBER)
inspect_clip(selected_clip)


## 3. Whisper Test

Compare language settings on the same audio without providing reference transcripts as prompts. This notebook tests only Whisper Large V3 Turbo.


### 3.1 Load Whisper Large V3 Turbo

Load the pinned model and processor on the GPU in FP16, then prepare independent generation configurations. See the [Whisper documentation](https://huggingface.co/docs/transformers/v4.57.1/model_doc/whisper).

Processor files are downloaded from the same pinned revision and loaded locally to avoid an optional additional_chat_templates lookup. Model weights, decoding settings, and clip identities are unchanged.


In [ ]:
# Load locally so tokenizer setup does not query Whisper's absent chat-template folder.
processor_dir = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    allow_patterns=[
        'config.json', 'preprocessor_config.json', 'tokenizer*.json',
        'vocab.json', 'merges.txt', 'normalizer.json',
        'added_tokens.json', 'special_tokens_map.json',
    ],
)
processor = AutoProcessor.from_pretrained(processor_dir, local_files_only=True)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE, use_safetensors=True, attn_implementation='sdpa'
).to(DEVICE).eval()
base_generation = copy.deepcopy(model.generation_config)
id_to_language = {token_id: token for token, token_id in base_generation.lang_to_id.items()}


### 3.2 Prepare the Selected Audio for Whisper

Retain your input shape/rate/type inspection. The processor converts the checked mono 16 kHz waveform into model features and an attention mask without receiving the reference transcript.


In [ ]:
audio_array = selected_clip['audio']
sample_rate = selected_clip['identity']['sample_rate']
print('Audio type:', type(audio_array), '| shape:', audio_array.shape, '| rate:', sample_rate)
example_inputs = processor(audio_array, sampling_rate=sample_rate, return_tensors='pt', return_attention_mask=True)
print('Processor keys:', list(example_inputs.keys()))
print('Input feature shape:', tuple(example_inputs.input_features.shape))
print('Attention-mask shape:', tuple(example_inputs.attention_mask.shape))


### 3.3 Compare Automatic Detection and Forced Nepali

Retain forced-Nepali transcription and compare it with automatic language detection on the same five clips. Use greedy decoding and a fresh configuration for each call; explicitly clear inherited language/forced decoder settings. Reset result lists when rerunning this cell.

Request the full generation result: the default Whisper tensor can omit prefix/EOS tokens, so inspecting its last token produced an unreliable truncation warning in the earlier draft. This version retains the raw language/EOS tokens. Printed WER/CER remain preliminary and include script differences.

Processor files are downloaded from the same pinned revision and loaded locally to avoid an optional additional_chat_templates lookup. Model weights, decoding settings, and clip identities are unchanged.


In [ ]:
mode_configs = {}
predictions = []

for mode, language in [('auto', None), ('forced_ne', 'ne')]:
    config = copy.deepcopy(base_generation)
    config.language = language
    config.task = 'transcribe'
    config.forced_decoder_ids = None
    config.do_sample = False
    config.num_beams = 1
    config.max_new_tokens = MAX_NEW_TOKENS
    mode_configs[mode] = config.to_dict()
    for clip in clips:
        inputs = processor(clip['audio'], sampling_rate=16000, return_tensors='pt', return_attention_mask=True)
        inputs = inputs.to(DEVICE, dtype=DTYPE)
        with torch.inference_mode():
            generated = model.generate(
                **inputs, generation_config=copy.deepcopy(config),
                return_timestamps=False, temperature=0.0, return_dict_in_generate=True
            )
        token_ids = generated.sequences[0].detach().cpu().tolist()
        hypothesis = processor.batch_decode(generated.sequences, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        reference = clip['identity']['transcription']
        normalized_reference = normalize_text(reference)
        normalized_hypothesis = normalize_text(hypothesis)
        record = {
            **clip['identity'], 'mode': mode, 'hypothesis': hypothesis,
            'normalized_reference': normalized_reference, 'normalized_hypothesis': normalized_hypothesis,
            'wer': wer(normalized_reference, normalized_hypothesis),
            'cer': cer(normalized_reference.replace(' ', ''), normalized_hypothesis.replace(' ', '')),
            'language_token': next((id_to_language[token_id] for token_id in token_ids[:4] if token_id in id_to_language), None),
            'ended_with_eos': token_ids[-1] == processor.tokenizer.eos_token_id,
            'generated_token_ids': token_ids,
        }
        predictions.append(record)
        print(f'\n{mode} | {record["clip_id"]} | CMI {record["cmi"]}')
        print('REFERENCE:', reference)
        print('WHISPER:  ', hypothesis)
        print(f'WER: {record["wer"]:.1%} | CER: {record["cer"]:.1%}')
        if not record['ended_with_eos']:
            print('CHECK: output did not end with EOS; the token limit may have truncated it.')


## 4. Pilot Results

Summarize this small diagnostic and preserve the outputs needed for later review. Listen before deciding whether a mismatch is transliteration, an incorrect word, an omission, or a hallucination.


### 4.1 Summarize WER and CER by Language Mode

Aggregate across all selected clips using total reference words/characters, rather than averaging clip error rates. Scores are fractions: 0.25 means 25%, and WER can exceed 100%. Five clips and silver references are insufficient to infer a CMI trend.

Both conditions must contain the same unique clip identities, not merely the same number of rows.


In [ ]:
assert len(predictions) == 2 * len(clips)
summary = {}
for mode in mode_configs:
    rows = [row for row in predictions if row['mode'] == mode]
    if len(rows) != len(clips) or {row['clip_id'] for row in rows} != {clip['identity']['clip_id'] for clip in clips}:
        raise ValueError(f'The {mode} condition does not cover the same unique CS clips.')
    references = [row['normalized_reference'] for row in rows]
    hypotheses = [row['normalized_hypothesis'] for row in rows]
    summary[mode] = {
        'clips': len(rows), 'reference_words': sum(len(text.split()) for text in references),
        'wer': wer(references, hypotheses),
        'cer': cer([text.replace(' ', '') for text in references], [text.replace(' ', '') for text in hypotheses]),
        'outputs_without_eos': sum(not row['ended_with_eos'] for row in rows),
    }
print('\nCORPUS SCORES — five-clip pilot only:')
print(json.dumps(summary, ensure_ascii=False, indent=2))


### 4.2 Export and Download the Results

The ZIP contains `results.json` (raw/normalized transcripts, clip and corpus scores, provenance, CMI, audio hashes, token IDs, and generation settings) and `environment.txt` (package snapshot and driver/GPU information). Keep the archive with your research outputs and share it for follow-up analysis. It contains no audio or weights.

Exact package/CUDA combinations may not reproduce bit-for-bit on another GPU. Full test evaluation, the two Nepali-specialized models, and formal CMI analysis belong in later notebooks after this pilot runs successfully.

The manifest also preserves all three reference renderings. Only canonical transcription is scored; the exploratory preview is documented separately from the scored CS manifest.


In [ ]:
created_at = datetime.now(timezone.utc)
bundle = {
    'created_at_utc': created_at.isoformat(), 'purpose': 'validation pilot, not final benchmark',
    'dataset': {'id': DATASET_ID, 'revision': DATASET_REVISION, 'config': 'default', 'split': SPLIT},
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION},
    'selection': {'speech_type': 'cs', 'seed': SEED, 'count': SAMPLE_COUNT, 'shuffle_buffer': SHUFFLE_BUFFER},
    'normalization': 'NFC; lowercase; Devanagari digits to ASCII; Unicode punctuation to spaces; collapse whitespace',
    'cer_units': 'Unicode code points excluding spaces',
    'runtime': {'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__,
                'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'dtype': str(DTYPE)},
    'generation_configs': mode_configs,
    'generation_overrides': {'return_timestamps': False, 'temperature': 0.0, 'return_dict_in_generate': True},
    'exploration': {'unfiltered_preview_count': len(unfiltered_preview), 'selected_cs_sample_number': SAMPLE_NUMBER},
    'manifest': [clip['identity'] for clip in clips], 'predictions': predictions, 'summary': summary,
}
environment = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
environment += '\n\n' + subprocess.check_output(['nvidia-smi'], text=True)
archive_path = Path('whisper_pilot_' + created_at.strftime('%Y%m%dT%H%M%S%fZ') + '.zip')
with zipfile.ZipFile(archive_path, mode='x', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr('results.json', json.dumps(bundle, ensure_ascii=False, indent=2, allow_nan=False))
    archive.writestr('environment.txt', environment)
print('Saved:', archive_path.resolve())
from google.colab import files
files.download(str(archive_path))


## Original Work Included and Corrections Made

Original cell numbers start at 0, as in the source notebook.

| Original cells | Included here |
| --- | --- |
| 0–2 | Dependencies, imports, PyTorch/GPU checks |
| 3–6 | Dataset/schema, first-record keys/metadata, three transcript formats |
| 7–8, 10–12 | Audio decoding, mono conversion, shape/type/rate/amplitude checks, playback, Whisper input preparation |
| 9, 13 | Whisper loading and forced-Nepali transcription |
| 14–17 | Unfiltered five-record preview and selectable audio inspection |
| 18–19 | Seeded CS-only sampling, transcript/CMI inspection, sample-number replay |

Repeated extraction/imports are combined. First-record example and filtered selected_clip stay distinct. One audio helper keeps each waveform paired with its reference. The CS filter comment matches its implementation. The data/model revisions are pinned, and validation replaces further test exploration. Results reset on rerun, and full generation outputs make language/EOS inspection reliable.

The original saved notebook contains one forced-Nepali output, but no saved automatic-language run or calculated WER/CER. Those are additions here, not invented earlier results. Your original notebook remains unchanged.

After this pilot, inspect the exported results before moving to the frozen test benchmark and the two Nepali-specialized models. No translation, training, or application work is included.